# 🥚 Egg Yolk Color Prediction - Model Training & Evaluation Notebook

สมุดบันทึกประเมินและเปรียบเทียบประสิทธิภาพของโมเดล Machine Learning 5 ตัว
ด้วยวิธี **Stratified 5-Fold Cross-Validation** (รักษาสัดส่วนคลาส 80/20)

คำนวณและแสดงผลตัววัดผล: **Train R², Test R², Train MAE, Test MAE, Train RMSE, Test RMSE, Train ±1 Acc, Test ±1 Acc** และ **Per-Class Accuracy Breakdown**

In [1]:
# 1. Import Libraries
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
# 2. Load Features & Stratified 5-Fold Cross-Validation Evaluation
features_csv = 'data/features.csv'
df = pd.read_csv(features_csv)
print(f"Loaded features dataset: {len(df)} samples across 12 classes.\n")

feature_cols = ['r', 'g', 'b', 'l', 'a', 'b_lab']
X = df[feature_cols].values
y = df['fan_score'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'SVR (RBF Kernel)': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', SVR(kernel='rbf', C=10.0, epsilon=0.1))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge(alpha=1.0))
    ])
}

comparison_results = []
per_class_results = []
classes = sorted(np.unique(y))

for name, pipeline in models.items():
    tr_r2, te_r2 = [], []
    tr_mae, te_mae = [], []
    tr_rmse, te_rmse = [], []
    tr_acc1, te_acc1 = [], []

    all_true, all_pred = [], []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[val_idx], y[val_idx]

        pipeline.fit(X_tr, y_tr)

        # Predict Train
        p_tr = pipeline.predict(X_tr)
        tr_r2.append(r2_score(y_tr, p_tr))
        tr_mae.append(mean_absolute_error(y_tr, p_tr))
        tr_rmse.append(np.sqrt(mean_squared_error(y_tr, p_tr)))
        tr_acc1.append(np.mean(np.abs(p_tr - y_tr) <= 1.0) * 100.0)

        # Predict Test (Validation)
        p_te = pipeline.predict(X_te)
        te_r2.append(r2_score(y_te, p_te))
        te_mae.append(mean_absolute_error(y_te, p_te))
        te_rmse.append(np.sqrt(mean_squared_error(y_te, p_te)))
        te_acc1.append(np.mean(np.abs(p_te - y_te) <= 1.0) * 100.0)

        all_true.extend(y_te)
        all_pred.extend(p_te)

    comparison_results.append({
        'Model': name,
        'Train R2': round(np.mean(tr_r2), 4),
        'Test R2': round(np.mean(te_r2), 4),
        'Train MAE': round(np.mean(tr_mae), 4),
        'Test MAE': round(np.mean(te_mae), 4),
        'Train RMSE': round(np.mean(tr_rmse), 4),
        'Test RMSE': round(np.mean(te_rmse), 4),
        'Train +/-1 Acc (%)': f"{np.mean(tr_acc1):.1f}%",
        'Test +/-1 Acc (%)': f"{np.mean(te_acc1):.1f}%"
    })

    # Calculate per-class metrics on Test set
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    rounded_pred = np.round(all_pred).astype(int)

    for c in classes:
        mask = (all_true == c)
        exact_acc = np.mean(rounded_pred[mask] == c) * 100.0
        pm1_acc = np.mean(np.abs(all_pred[mask] - c) <= 1.0) * 100.0
        
        per_class_results.append({
            'Model': name,
            'Class (Fan Score)': c,
            'Samples': int(np.sum(mask)),
            'Exact Acc (%)': round(exact_acc, 1),
            '+/-1 Acc (%)': round(pm1_acc, 1)
        })

res_df = pd.DataFrame(comparison_results).sort_values(by='Test R2', ascending=False)
print('=' * 90)
print('MODEL TRAIN vs TEST COMPARISON RESULTS (Stratified 5-Fold Cross-Validation)')
print('=' * 90)
print(res_df.to_string(index=False))
print('=' * 90)
print(f"\nBest Performing Model: {res_df.iloc[0]['Model']} (Test R^2 = {res_df.iloc[0]['Test R2']})")

Loaded features dataset: 647 samples across 12 classes.

MODEL TRAIN vs TEST COMPARISON RESULTS (Stratified 5-Fold Cross-Validation)
            Model  Train R2  Test R2  Train MAE  Test MAE  Train RMSE  Test RMSE Train +/-1 Acc (%) Test +/-1 Acc (%)
 SVR (RBF Kernel)    0.9173   0.9004     0.6233    0.6967      0.8571     0.9393              77.9%             74.2%
Gradient Boosting    0.9612   0.8900     0.4474    0.7206      0.5870     0.9875              91.2%             76.5%
    Random Forest    0.9847   0.8853     0.2665    0.7326      0.3693     1.0059              98.0%             75.3%
Linear Regression    0.8494   0.8448     0.8916    0.8978      1.1570     1.1729              64.8%             64.0%
 Ridge Regression    0.8276   0.8246     0.9788    0.9829      1.2381     1.2477              60.7%             60.9%

Best Performing Model: SVR (RBF Kernel) (Test R^2 = 0.9004)


In [3]:
# 3. Display Per-Class Accuracy Breakdown for Best Model (SVR)
per_class_df = pd.DataFrame(per_class_results)
top_model_name = res_df.iloc[0]['Model']
top_per_class = per_class_df[per_class_df['Model'] == top_model_name]

print(f"=== PER-CLASS ACCURACY BREAKDOWN FOR TOP MODEL ({top_model_name}) ===")
print(top_per_class.to_string(index=False))
print('=' * 90)

=== PER-CLASS ACCURACY BREAKDOWN FOR TOP MODEL (SVR (RBF Kernel)) ===
           Model  Class (Fan Score)  Samples  Exact Acc (%)  +/-1 Acc (%)
SVR (RBF Kernel)                  4       36           75.0          86.1
SVR (RBF Kernel)                  5       41           36.6          56.1
SVR (RBF Kernel)                  6       49           34.7          57.1
SVR (RBF Kernel)                  7       42           40.5          64.3
SVR (RBF Kernel)                  8       64           56.2          89.1
SVR (RBF Kernel)                  9      105           50.5          76.2
SVR (RBF Kernel)                 10       99           49.5          73.7
SVR (RBF Kernel)                 11       75           46.7          84.0
SVR (RBF Kernel)                 12       33           39.4          69.7
SVR (RBF Kernel)                 13       21           33.3          71.4
SVR (RBF Kernel)                 14       33           30.3          57.6
SVR (RBF Kernel)                 15       